# ♟️ Ultralytics YOLO26n - 2D Chess Piece & Board Detection
### End-to-End Dataset Generation, Quality Assurance, Training & Export on Kaggle

This notebook provides an automated workflow to generate high-quality data and train **Ultralytics YOLO26n (`yolo26n.pt`)** to detect 2D chess pieces and boards.

- **YOLO26 Architecture:** Native end-to-end NMS-free head, MuSGD optimizer, ProgLoss progressive supervision, and STAL small-target-aware label assignment.
- **Domain-Specific Augmentations:** Orientation-preserving transforms, realistic square highlights, check indicators, eval bars, clocks, and screen captures.
- **Zero Data Leakage:** Piece sets and board themes are partitioned deterministically so validation and test splits evaluate novel, unseen styles.
- **Kaggle GPU Optimized:** Automatic device selection, session timeout protection, and direct artifact packaging.

In [ ]:
# 1. Install Ultralytics and dependencies
!pip install -q "ultralytics>=8.3.0"

import os
import time
import torch
import ultralytics

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"Device Count: {torch.cuda.device_count()}")
print(f"Ultralytics Version: {ultralytics.__version__}")

In [ ]:
# 2. Clone or update the project repository
REPO_DIR = "/kaggle/working/2d-chess-pieces-detection"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Zai-Kun/2d-chess-pieces-detection {REPO_DIR}
else:
    print(f"Repository already present at: {REPO_DIR}")

%cd {REPO_DIR}

## 2. Dataset Generation & Quality Assurance (QA)

Generate a diverse synthetic-to-real dataset:
- **Diverse Contexts:** Full-board, Web/App UI (dark/light themes, eval bar, player cards, timers), embedded boards on cluttered scenes, and photographed screens.
- **Strict Bounding Boxes:** Sub-pixel alpha thresholding (`alpha >= 20`) with baseline-aligned pieces.
- **Reproducible Zero-Leakage Splitting:** Disjoint partition of piece sets (66), boards (112), and backgrounds (2,900+).

In [ ]:
# 3. Generate high-quality dataset
# Generates 3,000 train, 450 val, and 450 test samples (~2-4 minutes on Kaggle CPU/GPU)
import os
import subprocess

cmd = [
    "python3", "generate_datasets.py",
    "--train-count", "3000",
    "--val-count", "450",
    "--test-count", "450",
    "--imgsz", "640",
    "--seed", "42",
    "--clean"
]
ret = subprocess.run(cmd)
if ret.returncode != 0:
    raise RuntimeError(f"Dataset generation failed with return code {ret.returncode}! Stopping pipeline.")

if not (os.path.exists("datasets/images/train") and len(os.listdir("datasets/images/train")) > 0):
    raise RuntimeError("Training images not found after generation! Stopping pipeline.")


In [ ]:
# 4. Run Automated QA Suite
# Checks 1:1 pair integrity, coordinate ranges, boundary compliance, and duplicate prevention
import subprocess

ret = subprocess.run(["python3", "verify_dataset.py", "--data-dir", "datasets"])
if ret.returncode != 0:
    raise RuntimeError("Dataset verification failed QA checks! Stopping pipeline.")


In [ ]:
# 5. Display sample preview montage with bounding boxes
from PIL import Image
import matplotlib.pyplot as plt

preview_path = "results/previews/preview_train.jpg"
if os.path.exists(preview_path):
    plt.figure(figsize=(15, 10))
    plt.imshow(Image.open(preview_path))
    plt.axis("off")
    plt.title("Sample Generated Training Images with YOLO Bounding Boxes", fontsize=14)
    plt.show()

## 3. Train Ultralytics YOLO26n

Train `yolo26n.pt` using the recommended fine-tuning recipe:
- **Optimizer:** `MuSGD` with `lr0=0.01` and cosine annealing decay (`lrf=0.01`).
- **Domain Augmentations:** `fliplr=0.0`, `flipud=0.0`, `mixup=0.0`, `mosaic=0.2`, `close_mosaic=10`.
- **Session Timeout Protection:** A callback checks elapsed time and halts training gracefully before Kaggle's 12-hour limit.

In [ ]:
# 6. Train YOLO26n on Kaggle GPU
import os
import time
import yaml
import torch
from ultralytics import YOLO

# Ensure absolute dataset paths in chess_detection.yaml so Ultralytics resolves correctly
yaml_path = os.path.abspath("chess_detection.yaml")
with open(yaml_path, "r") as f:
    cfg = yaml.safe_load(f)

dataset_root = os.path.abspath("datasets")
cfg["path"] = dataset_root
cfg["train"] = "images/train"
cfg["val"] = "images/val"
cfg["test"] = "images/test"

with open(yaml_path, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(f"Verified dataset path: {cfg['path']}")
assert os.path.exists(os.path.join(dataset_root, "images", "val")), f"Missing validation images at: {dataset_root}/images/val"

# Select device
device = "0" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available() and torch.cuda.device_count() > 1:
    device = "0,1"
print(f"Training on device: {device}")

# Session timeout protection callback
start_time = time.time()

def kaggle_timeout_guard(trainer):
    elapsed_hours = (time.time() - start_time) / 3600
    if elapsed_hours > 10.5:
        print("\n⚠️ Approaching Kaggle session timeout limit! Saving checkpoint and stopping gracefully...")
        trainer.stop = True

# Load pretrained YOLO26n model
model = YOLO("yolo26n.pt")
model.add_callback("on_train_epoch_end", kaggle_timeout_guard)

# Execute training with recommended YOLO26 settings
train_results = model.train(
    data=yaml_path,
    epochs=60,
    imgsz=640,
    batch=16 if device != "0,1" else 32,
    device=device,
    optimizer="MuSGD",
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    patience=15,
    project="runs/detect",
    name="yolo26n_chess",
    save=True,
    save_period=5,
    plots=True,
    exist_ok=True,
    # Domain-tuned augmentations
    fliplr=0.0,
    flipud=0.0,
    mixup=0.0,
    mosaic=0.2,
    close_mosaic=10,
    degrees=0.0,
    scale=0.2,
    translate=0.1,
    hsv_h=0.015,
    hsv_s=0.30,
    hsv_v=0.30,
)


## 4. Evaluation & Generalization Benchmarking

Evaluate the best model weights on both the **validation** split and the **unseen hold-out test** split.
The test metrics measure true zero-shot generalization on piece sets and board themes never seen during training.

In [ ]:
# 7. Evaluate on Validation and Unseen Test Splits
best_weights = "runs/detect/yolo26n_chess/weights/best.pt"
best_model = YOLO(best_weights)

print("--- Validation Split Metrics ---")
val_metrics = best_model.val(data="chess_detection.yaml", split="val")
print(f"Val mAP50:    {val_metrics.box.map50:.4f}")
print(f"Val mAP50-95: {val_metrics.box.map:.4f}")

print("\n--- Unseen Test Split Metrics (Zero Data Leakage) ---")
test_metrics = best_model.val(data="chess_detection.yaml", split="test")
print(f"Test mAP50:    {test_metrics.box.map50:.4f}")
print(f"Test mAP50-95: {test_metrics.box.map:.4f}")

In [ ]:
# 8. Display training curves and confusion matrix
run_dir = "runs/detect/yolo26n_chess"
for plot_file in ["results.png", "confusion_matrix.png", "BoxPR_curve.png"]:
    full_path = os.path.join(run_dir, plot_file)
    if os.path.exists(full_path):
        plt.figure(figsize=(10, 6))
        plt.imshow(Image.open(full_path))
        plt.axis("off")
        plt.title(plot_file, fontsize=13)
        plt.show()

## 5. Visualizing Detections on Novel Test Images

In [ ]:
# 9. Run inference on random test samples and visualize predictions
import glob
import random

test_images = sorted(glob.glob("datasets/images/test/*.jpg"))
if test_images:
    selected = random.sample(test_images, min(4, len(test_images)))
    preds = best_model.predict(source=selected, imgsz=640, conf=0.25)

    fig, axes = plt.subplots(2, 2, figsize=(14, 14))
    for idx, res in enumerate(preds):
        ax = axes[idx // 2, idx % 2]
        ax.imshow(res.plot()[:, :, ::-1])  # BGR to RGB
        ax.axis("off")
        ax.set_title(f"Test Prediction {idx + 1} ({len(res.boxes)} objects)", fontsize=12)
    plt.tight_layout()
    plt.show()

## 6. Export to ONNX & Package Artifacts for Direct Download

Exports the model to ONNX format (featuring native NMS-free inference) and copies `best.pt`, `best.onnx`, and summary plots directly to `/kaggle/working/` for direct download from Kaggle's Output panel.

In [ ]:
# 10. Export to ONNX and copy artifacts to /kaggle/working root
import shutil

# Export to ONNX (with native end-to-end NMS-free inference)
onnx_path = best_model.export(format="onnx", imgsz=640, dynamic=False)
print(f"Exported ONNX model: {onnx_path}")

output_root = "/kaggle/working"
artifacts = [
    (best_weights, os.path.join(output_root, "yolo26n_chess_best.pt")),
    (onnx_path, os.path.join(output_root, "yolo26n_chess_best.onnx")),
    (os.path.join(run_dir, "results.png"), os.path.join(output_root, "training_results.png")),
    (os.path.join(run_dir, "confusion_matrix.png"), os.path.join(output_root, "confusion_matrix.png")),
]

print("\nPackaging outputs for direct Kaggle download:")
for src, dst in artifacts:
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"  -> Copied {os.path.basename(dst)} to {output_root}")

print("\n🎉 Done! All artifacts are ready in Kaggle's Output panel for download.")